# Intermediate Queries

Uses cleaned data from **../../Part 2/cleaned_data** and the SQLite database in **../database/ecommerce.db**.

In [1]:
"""
Week 8 Mini Project - E-Commerce Order Analytics System
Part 3: SQL Analysis - Batch 2 (Intermediate Queries)

  4. Customers who placed orders but never had any item delivered
  5. Products that were ordered but had more returns than purchases
  6. Return rate (returned items / total items) per category
"""

from pathlib import Path
import sqlite3
import pandas as pd

DB_PATH = Path("../database/ecommerce.db")

def run_query(conn, sql):
    return pd.read_sql_query(sql, conn)

def query_4_never_delivered_customers(conn):
    """
    'status' lives at the order level, not the item level, so "never had
    any item delivered" is read here as: this customer has placed at least
    one order, and none of their orders ever reached DELIVERED status.
    """
    sql = """
        SELECT
            CAST(o.customer_id AS INTEGER) AS customer_id,
            c.customer_name,
            COUNT(*) AS total_orders,
            GROUP_CONCAT(DISTINCT o.status) AS statuses_seen
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        WHERE o.customer_id IS NOT NULL
        GROUP BY o.customer_id, c.customer_name
        HAVING SUM(CASE WHEN o.status = 'DELIVERED' THEN 1 ELSE 0 END) = 0
        ORDER BY total_orders DESC;
    """
    return run_query(conn, sql)


def query_5_more_returns_than_purchases(conn):
    """
    A negative quantity row is a return; a positive one is a purchase.
    'More returns than purchases' is compared in units, not row counts -
    e.g. one purchase of 5 units offset by two returns of -3 each is a net
    return situation even though it's fewer rows.
    """
    sql = """
        SELECT
            oi.product_id,
            p.product_name,
            SUM(CASE WHEN oi.quantity > 0 THEN oi.quantity ELSE 0 END) AS purchased_units,
            SUM(CASE WHEN oi.quantity < 0 THEN -oi.quantity ELSE 0 END) AS returned_units
        FROM order_items oi
        JOIN products p ON oi.product_id = p.product_id
        GROUP BY oi.product_id, p.product_name
        HAVING returned_units > purchased_units
        ORDER BY returned_units DESC;
    """
    return run_query(conn, sql)


def query_6_return_rate_per_category(conn):
    """
    return_rate = returned units / total units moved (purchased + returned)
    in that category. Expressed as a percentage, rounded to 2 dp.
    """
    sql = """
        SELECT
            p.category,
            SUM(CASE WHEN oi.quantity > 0 THEN oi.quantity ELSE 0 END) AS purchased_units,
            SUM(CASE WHEN oi.quantity < 0 THEN -oi.quantity ELSE 0 END) AS returned_units,
            ROUND(
                100.0 * SUM(CASE WHEN oi.quantity < 0 THEN -oi.quantity ELSE 0 END)
                / NULLIF(SUM(ABS(oi.quantity)), 0),
                2
            ) AS return_rate_percent
        FROM order_items oi
        JOIN products p ON oi.product_id = p.product_id
        GROUP BY p.category
        ORDER BY return_rate_percent DESC;
    """
    return run_query(conn, sql)


def main():
    conn = sqlite3.connect(DB_PATH)

    print("\n=== Query 4: Customers Who Never Had Any Order Delivered ===")
    result_4 = query_4_never_delivered_customers(conn)
    print(f"({len(result_4)} customers match)")
    print(result_4.head(10).to_string(index=False))

    print("\n=== Query 5: Products With More Returns Than Purchases ===")
    result_5 = query_5_more_returns_than_purchases(conn)
    print(f"({len(result_5)} products match)")
    print(result_5.to_string(index=False))

    print("\n=== Query 6: Return Rate per Category ===")
    print(query_6_return_rate_per_category(conn).to_string(index=False))

    conn.close()


if __name__ == "__main__":
    main()



=== Query 4: Customers Who Never Had Any Order Delivered ===
(44 customers match)
 customer_id      customer_name  total_orders                     statuses_seen
         380      Bianca Carter             5 CANCELLED,PLACED,SHIPPED,RETURNED
          41      Jessica Smith             4           SHIPPED,PLACED,RETURNED
         235       Jason Parker             4 RETURNED,SHIPPED,PLACED,CANCELLED
         434      Becky Johnson             4           SHIPPED,RETURNED,PLACED
         474 Margaret Hernandez             4 RETURNED,CANCELLED,PLACED,SHIPPED
         591        Karen Perez             4         RETURNED,CANCELLED,PLACED
          30    Rachel Mitchell             3                    PLACED,SHIPPED
         140     Kristen Willis             3                  SHIPPED,RETURNED
         244       Jeremy Scott             3          PLACED,CANCELLED,SHIPPED
         443     Jennifer Doyle             3                 CANCELLED,SHIPPED

=== Query 5: Products With More Retu